In [ ]:
# import pandas as pd
# import os

# file_path = "../data/processed/market_data_clean_v2.parquet"

# d = pd.read_parquet(file_path)
# d.shape[1] #no. of columns
# d = d.drop(columns=['naew'])

66

## Testing the price model finaal

In [ ]:
import lightgbm as lgb
import numpy as np
from features import load_clean_data, get_src_cols, add_lags, get_feature_cols, cat_cols

booster = lgb.Booster(model_file='../models/price_surrogate_final.txt')

d = load_clean_data()
src_cols = get_src_cols(d)   # still needed, just to know which raw columns are the "sources" for feature_cols
feature_cols = get_feature_cols(src_cols)

sample = d[d['product_name'].isin(['Apple', 'Sugarcane', 'Orange_Sweet', 'Potato'])].copy()
sample = sample.sort_values(['product_name', 'month_idx'])

X_sample = sample[feature_cols].copy()
for c in cat_cols:
    X_sample[c] = X_sample[c].astype('category')

pred_log = booster.predict(X_sample)
sample['predicted_price'] = np.expm1(pred_log)
print(sample[['product_name', 'month_idx', 'avg_price', 'predicted_price']])

In [6]:
pd.set_option('display.max_rows', None)        # show all rows
pd.set_option('display.max_columns', None)     # show all columns
pd.set_option('display.expand_frame_repr', False)  # don't wrap with ...
pd.set_option('display.max_colwidth', None)    # show full column content


print("Missing avg (NaN or 0):", ((d['avg_price'].isna()) | (d['avg_price'] == 0)).sum())
print("Missing min (NaN or 0):", ((d['min_price'].isna()) | (d['min_price'] == 0)).sum())
print("Missing max (NaN or 0):", ((d['max_price'].isna()) | (d['max_price'] == 0)).sum())
print("Missing volume (NaN or 0):", ((d['volume'].isna()) | (d['volume'] == 0)).sum())
print("Missing total_sources (NaN or 0):", ((d['total_sources'].isna()) | (d['total_sources'] == 0)).sum())


missing_maxmin = d[d['min_price'].isna() | d['max_price'].isna()]
print(missing_maxmin['product_name'].value_counts())

missing_vol = d[d['volume'].isna() | (d['volume']==0)]
print(missing_vol['product_name'].value_counts())

Missing avg (NaN or 0): 0
Missing min (NaN or 0): 77
Missing max (NaN or 0): 76
Missing volume (NaN or 0): 84
Missing total_sources (NaN or 0): 85
product_name
Bayar             1
Fenugreek_Leaf    1
Grapes            1
Green_Ve          1
Haluwabed         1
Sarifa            1
Yam               1
Name: count, dtype: int64
product_name
Apple_Fuji        7
Banana            7
Tomato_Big        7
Potato_W          7
Kurilo            6
Potato_Blue       6
Red_Grapes        5
Bambooshoot       4
Brinjal_Gland     4
Orange_Sweet      2
Pear_China        2
Sajiwan           2
Bakula            2
Bayar             2
Garlic_Dry        2
Turnip            2
Sponge_Gourd      2
Jack_Fruit        1
Guava             1
Garlic_Green      1
Fenugreek_Leaf    1
Cress_Leaf        1
Broccoli          1
Amala             1
Akabary_Chilly    1
Gundruk           1
Radish_Red        1
Pumpkin_Leaf      1
Strawberry        1
Sarifa            1
Sweet_Potato      1
Tomato_Small      1
Name: count, dtype: i

In [7]:
for p in ['Apple_Fuji','Banana','Tomato_Big','Potato_W','Kurilo','Potato_Blue']:
    print(p, d[d['product_name']==p]['month_idx'].nunique(), 'total months present,',
          d[(d['product_name']==p) & (d['volume'].isna() | (d['volume']==0))]['month_idx'].nunique(), 'missing volume')

zero_vol = d[(d['volume'].isna()) | (d['volume']==0)]
print(zero_vol[['product_name','month_idx','volume','total_sources','avg_price']].head(20))

Apple_Fuji 7 total months present, 7 missing volume
Banana 7 total months present, 7 missing volume
Tomato_Big 7 total months present, 7 missing volume
Potato_W 7 total months present, 7 missing volume
Kurilo 8 total months present, 6 missing volume
Potato_Blue 6 total months present, 6 missing volume
      product_name  month_idx  volume  total_sources  avg_price
2   Akabary_Chilly          8       0              0     555.56
13           Amala         10       0              0     145.00
24      Apple_Fuji          1       0              0     289.20
25      Apple_Fuji          3       0              0     300.00
26      Apple_Fuji          5       0              0     300.00
27      Apple_Fuji          7       0              0     300.00
28      Apple_Fuji          8       0              0     300.00
29      Apple_Fuji          9       0              0     300.00
30      Apple_Fuji         10       0              0     300.00
51          Bakula          7       0              0     

In [8]:
file_path2 = "../data/processed/market_data_clean_v3.parquet"
v3_d = pd.read_parquet(file_path2)
print(v3_d['min_price'].isna().sum(), v3_d['max_price'].isna().sum())
print(
    "Zero counts:",
    (v3_d['min_price'] == 0).sum(),
    (v3_d['max_price'] == 0).sum()
)

print(v3_d.nlargest(5, 'volatility')[['product_name','month_idx','min_price','max_price','avg_price','volatility']])

p = v3_d.nlargest(1, 'volatility')['product_name'].values[0]
print(v3_d[v3_d['product_name'] == p][['month_idx','product_name','min_price','max_price','avg_price','volatility']])


0 0
Zero counts: 0 0
         product_name  month_idx  min_price  max_price  avg_price  volatility
191  Cauliflower_Hill          7       10.0      140.0      52.36    2.482811
247          Cucumber          1       25.0      150.0      53.00    2.358491
231   Coriander_Green          7       20.0      150.0      59.29    2.192613
236           Cow_Pea          2       30.0      180.0      68.47    2.190740
233   Coriander_Green          9       30.0      250.0     102.05    2.155806
     month_idx      product_name  min_price  max_price  avg_price  volatility
185          1  Cauliflower_Hill       70.0      150.0      95.00    0.842105
186          2  Cauliflower_Hill       60.0      150.0      83.06    1.083554
187          3  Cauliflower_Hill      100.0      200.0     148.13    0.675083
188          4  Cauliflower_Hill      120.0      180.0     138.42    0.433463
189          5  Cauliflower_Hill       60.0      150.0      98.42    0.914448
190          6  Cauliflower_Hill       60.0

In [9]:
import numpy as np
# in 003risk_features.py, where you already compute this — save it instead of just using inline
vol_stats = v3_d.groupby('product_name')['total_sources'].agg(['mean', 'std'])
vol_stats['volume_cv'] = np.where(vol_stats['mean'] > 0, vol_stats['std'] / vol_stats['mean'], np.nan)
vol_stats[['volume_cv']].to_parquet('../data/processed/product_volume_stats.parquet')

In [10]:
import json
with open('../data/processed/risk_scaling_ref.json') as f:
    scaling_ref = json.load(f)

product_vol_stats = pd.read_parquet('../data/processed/product_volume_stats.parquet')
print(product_vol_stats.head())
print(product_vol_stats.index.name)

                volume_cv
product_name             
Akabary_Chilly   1.384125
Amala            0.614161
Apple            0.539025
Apple_Fuji            NaN
Arum             1.101273
product_name


In [11]:
from features import load_clean_data, compute_risk
d_v2 = load_clean_data()  # v2
result = compute_risk(d_v2, product_vol_stats, scaling_ref)
print(result[['product_name','month_idx','risk_score','risk']].head(10))

     product_name  month_idx  risk_score    risk
0  Akabary_Chilly          6    0.269647     Low
1  Akabary_Chilly          7    0.460101  Medium
2  Akabary_Chilly          8    0.272810     Low
3  Akabary_Chilly          9    0.497291  Medium
4  Akabary_Chilly         10    0.278951     Low
5           Amala          2    0.540519  Medium
6           Amala          3    0.391777  Medium
7           Amala          4    0.317717     Low
8           Amala          5    0.356485  Medium
9           Amala          6    0.424503  Medium


In [12]:
v3 = pd.read_parquet('../data/processed/market_data_clean_v3.parquet')
compare = result.merge(v3[['product_name','month_idx','risk_score']], on=['product_name','month_idx'], suffixes=('_new','_v3'))
print((compare['risk_score_new'] - compare['risk_score_v3']).abs().max())  # should be ~0

0.0


In [13]:
print(v3_d['risk_score'].quantile([0.333, 0.667]))

0.333    0.177329
0.667    0.318698
Name: risk_score, dtype: float64


In [16]:
v3_d[v3_d['product_name']=='Orange_Sweet']['india'].pct_change().describe()


c:\Users\poude\Desktop\SATRI_DA_ML_july\venv\lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\poude\Desktop\SATRI_DA_ML_july\venv\lib\site-packages\numpy\lib\_function_base_impl.py:4655: RuntimeWarning: invalid value encountered in multiply
  lerp_interpolation = asanyarray(add(a, diff_b_a * t, out=out))


count    9.000000
mean          inf
std           NaN
min     -1.000000
25%     -1.000000
50%     -0.050584
75%           NaN
max           inf
Name: india, dtype: float64

In [32]:
# def get_product_baseline(v3_d, product, col):
#     rows = v3_d[(v3_d['product_name'] == product) & (v3_d[col] > 0)]
#     if len(rows) == 0:
#         return None  # no historical sourcing from this column at all
#     return rows[col].mean()


# for product in v3_d['product_name'].unique():
#     print(f"{product}", get_product_baseline(v3_d, product, 'india') )
nonzero_india_rows = v3_d[v3_d['india'] > 0]
print(nonzero_india_rows[['product_name','month_idx','india']].head(10))

      product_name  month_idx   india
3   Akabary_Chilly          9    5260
4   Akabary_Chilly         10    3970
5            Amala          3   18785
6            Amala          4   17215
7            Amala          5   22660
8            Amala          6   28685
9            Amala          7   15340
10           Amala          8   12730
11           Amala          9    5010
13           Apple          1  358500


In [29]:

for product in v3_d['product_name'].unique():
    print(f"{product}", get_product_baseline(v3_d, product, 'local') )

Akabary_Chilly None
Amala None
Apple None
Apple_Fuji None
Arum 6698.888888888889
Avocado 1157.7142857142858
Bakula 10497.5
Bambooshoot 7568.333333333333
Banana None
Banana_Green 2410.0
Barela 5531.666666666667
Bayar 45.0
Beat_Chukunder 2165.2
Bitter_Gourd 31437.428571428572
Bottle_Gourd 18684.0
Brinjal_Gland None
Brinjal_Long 18566.0
Broad_Leaf_Mustard 41073.5
Broccoli 35233.28571428572
Cabbage 81419.66666666667
Cabbage_Red 3427.4
Carrot 17994.166666666668
Cauliflower_Hill 98773.75
Chilly_Green 18765.9
Christophine 10491.0
Coconut None
Coriander_Green 26981.3
Cow_Pea 23258.88888888889
Cress_Leaf 3840.0
Cucumber 53609.3
Dragon_Fruits 48.333333333333336
Fenugreek_Leaf None
French_Bean 30748.4
Garlic_Dry None
Garlic_Green 2471.5555555555557
Ginger 7004.1
Grapes None
Green_Peas 12688.333333333334
Guava 1176.75
Gundruk 30219.0
Haluwabed None
Jack_Fruit 2380.0
Khasa_Garlic_Dry None
Kiwi 135.0
Kurilo 68.0
Lemon 1416.6666666666667
Malta None
Mango None
Mombin 412.5
Mushroom 30481.5
Okara 25052